In [ ]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string

import numpy as np

In [7]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\msara\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\msara\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\msara\AppData\Roaming\nltk_data...


True

In [8]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [2]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
379,I really enjoy this movie. The first time it w...,positive
940,For me personally this film goes down in my to...,positive
210,<br /><br />Robot jox is a great little film o...,positive
201,"Blue ribbon banners, stars and stripes forever...",positive
527,Here is another great film critics will love. ...,negative


In [3]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

<>:32: SyntaxWarning: invalid escape sequence '\s'
<>:32: SyntaxWarning: invalid escape sequence '\s'
C:\Users\msara\AppData\Local\Temp\ipykernel_38948\3798096103.py:32: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


In [9]:
df = normalize_text(df)
df.head()

,review,sentiment
379,really enjoy movie first time turner classic m...,positive
940,personally film go top four time exception jam...,positive
210,br br robot jox great little film ok set bad a...,positive
201,blue ribbon banner star stripe forever decorat...,positive
527,another great film critic love problem good mo...,negative


In [10]:
df['sentiment'].value_counts()

sentiment
negative    258
positive    242
Name: count, dtype: int64

In [11]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [12]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
379,really enjoy movie first time turner classic m...,1
940,personally film go top four time exception jam...,1
210,br br robot jox great little film ok set bad a...,1
201,blue ribbon banner star stripe forever decorat...,1
527,another great film critic love problem good mo...,0


In [13]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [14]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [16]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/ravigupta5/mlops_proj.mlflow')
dagshub.init(repo_owner='ravigupta5', repo_name='mlops_proj', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=b95ba26b-2519-424c-82e8-5bd8257d0f41&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=136bb92056dbcf77b7e9f1429944692a24bad9f5b1d485751ecc70f205504518




Accessing as ravigupta5

Initialized MLflow to track repo "ravigupta5/mlops_proj"

Repository ravigupta5/mlops_proj initialized!

2026/07/04 07:40:31 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/569e3124b2944077b3c431637399ff69', creation_time=1783131030005, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1783131030005, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [17]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-07-04 07:41:53,830 - INFO - Starting MLflow run...
2026-07-04 07:41:54,721 - INFO - Logging preprocessing parameters...
2026-07-04 07:41:55,738 - INFO - Initializing Logistic Regression model...
2026-07-04 07:41:55,738 - INFO - Fitting the model...
2026-07-04 07:41:55,782 - INFO - Model training complete.
2026-07-04 07:41:55,784 - INFO - Logging model parameters...
2026-07-04 07:41:56,104 - INFO - Making predictions...
2026-07-04 07:41:56,114 - INFO - Calculating evaluation metrics...
2026-07-04 07:41:56,118 - INFO - Logging evaluation metrics...
2026-07-04 07:41:57,371 - INFO - Saving and logging the model...
2026/07/04 07:41:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-07-04 07:42:32,800 - INFO - Model training and logging completed in 38.08 seconds.
2026-07-04 07:42:32,809 - INFO - Accuracy: 0.664
2026-07-04 07:42:32,809 - INFO - Precision: 0.7222222222222222
2026-07-04 07:42:32,809 - INFO - Recall: 0.5909090909090909
2026-07-04

🏃 View run zealous-robin-303 at: https://dagshub.com/ravigupta5/mlops_proj.mlflow/#/experiments/0/runs/ac86356bccc7447db4c8064f54e5ee4f
🧪 View experiment at: https://dagshub.com/ravigupta5/mlops_proj.mlflow/#/experiments/0


In [18]:
1+2

3